# 3a — Sample Validation Set

Constructs the human validation sample described in Stage 3.1 of `research_project_plan_v2.md`.

**Inputs:**
- `data/chunk_registry_v1.parquet` or `data/chunk_registry_v1.csv`
- Pass-1 JSON outputs under `outputs/` (optional but used for rare-event oversampling)

**Outputs:**
- `data/chunk_registry_v2.parquet` or CSV fallback
- updated validation flags: `human_validation_set`, `utterance_validation_set`, `oversampled_for`


## Step 0 — Imports and paths

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path().resolve()
ANALYSIS_V2 = NOTEBOOK_DIR.parent
PROJECT_ROOT = ANALYSIS_V2.parent
DATA_DIR = ANALYSIS_V2 / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'

DATA_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR  :', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


## Step 1 — Load registry and define validation tiers

In [ ]:
def load_table(stem: str) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{stem}.parquet'
    csv_path = DATA_DIR / f'{stem}.csv'
    if parquet_path.exists():
        try:
            return pd.read_parquet(parquet_path)
        except Exception as e:
            print(f'Falling back to CSV for {stem}: {type(e).__name__}: {e}')
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'Could not find {parquet_path.name} or {csv_path.name}')


registry = load_table('chunk_registry_v1').copy()

if 'human_validation_set' not in registry.columns:
    registry['human_validation_set'] = False
if 'utterance_validation_set' not in registry.columns:
    registry['utterance_validation_set'] = False
if 'oversampled_for' not in registry.columns:
    registry['oversampled_for'] = pd.NA

TIER_1 = [
    'idea_trajectory',
    'collective_engagement_level',
    'explicit_commitment_signal',
    'decision_crystallization_level',
    'pronoun_shift_flag',
    'cross_disciplinary_bridging',
    'shared_vision_indicator',
]
TIER_2 = [
    'problem_specificity_level',
    'ambition_level',
    'laughter_quality',
    'dissent_response_quality',
    'risk_acknowledgment_with_enthusiasm',
    'personal_disclosure',
    'meeting_structure_quality',
]
TIER_3 = [
    'screenshare_active',
    'artifact_interaction',
    'funding_awareness_signal',
    'prior_relationship_signal',
    'explicit_complementarity_recognition',
    'skill_gap_identification',
]
UTTERANCE_PRIORITY = [
    'Idea_Management',
    'Integration_Practices',
    'Pronoun_Framing',
    'interruption_type',
]

print(f'Registry: {len(registry):,} chunks across {registry["conference_id"].nunique()} conferences')
registry.head(2)


## Step 2 — Stratified chunk-level sample

In [ ]:
registry['strat_key'] = (
    registry['conference_id'].astype(str) + '__' +
    registry['chunk_position'].astype(str) + '__' +
    registry['outcome_has_funded_teams'].astype(str)
)

key_counts = registry['strat_key'].value_counts()
rare_keys = key_counts[key_counts < 2].index
registry['strat_key_collapsed'] = registry['strat_key'].where(
    ~registry['strat_key'].isin(rare_keys),
    registry['conference_id'].astype(str) + '__' + registry['outcome_has_funded_teams'].astype(str)
)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
_, val_idx = next(sss.split(registry, registry['strat_key_collapsed']))
registry['human_validation_set'] = False
registry.loc[registry.index[val_idx], 'human_validation_set'] = True

print(registry['human_validation_set'].value_counts())
display(
    registry.loc[registry['human_validation_set']]
    .groupby(['conference_id', 'chunk_position', 'outcome_has_funded_teams'])
    .size()
    .reset_index(name='n')
    .sort_values(['conference_id', 'chunk_position', 'outcome_has_funded_teams'])
)


## Step 3 — Optional rare-event oversampling from AI chunk summaries

In [ ]:
def normalize_filename(name: str) -> str:
    name = re.sub(r'[^a-zA-Z0-9._()]', '_', name)
    name = re.sub(r'_+', '_', name)
    return name.strip('_')


def resolve_json_path(row: pd.Series) -> Path:
    conf = row['conference_id']
    session_key = row['session_key']
    chunk_fn = row['chunk_file_name']
    stem_norm = normalize_filename(Path(chunk_fn).stem)
    session_grp = session_key.split('/')[0].replace('-', '_')
    out_dir = OUTPUT_DIR / conf / f'output_{session_grp}'
    if '/' in session_key:
        recording_folder = session_key.split('/', 1)[1]
        base = out_dir / recording_folder
    else:
        base = out_dir
    plain = base / f'{stem_norm}.json'
    attn = base / f'ATTN_{stem_norm}.json'
    return plain if plain.exists() else attn


def load_all_chunk_summaries(registry_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(registry_df.iterrows(), total=len(registry_df), desc='Loading AI summaries', unit='chunk'):
        path = resolve_json_path(row)
        if not path.exists():
            continue
        try:
            with open(path) as f:
                data = json.load(f)
        except Exception:
            continue
        summary = data.get('chunk_summary', {}).copy()
        summary['chunk_id'] = row['chunk_id']
        rows.append(summary)
    return pd.DataFrame(rows)


RARE_FLAGS = [
    'explicit_commitment_signal',
    'cross_disciplinary_bridging',
    'risk_acknowledgment_with_enthusiasm',
]
MIN_POSITIVE = 15

ai_summaries = load_all_chunk_summaries(registry)
if ai_summaries.empty:
    print('No AI chunk summaries found. Skipping rare-event oversampling.')
else:
    for flag in RARE_FLAGS:
        if flag not in ai_summaries.columns:
            print(f'Skipping {flag}: not found in AI summaries')
            continue
        n_positive_in_val = (
            ai_summaries.loc[
                ai_summaries['chunk_id'].isin(registry.loc[registry['human_validation_set'], 'chunk_id']),
                flag
            ]
            .eq('Yes')
            .sum()
        )
        if n_positive_in_val >= MIN_POSITIVE:
            print(f'{flag}: already has {n_positive_in_val} positives in validation set')
            continue
        positive_not_in_val = ai_summaries.loc[
            (ai_summaries[flag] == 'Yes') &
            (~ai_summaries['chunk_id'].isin(registry.loc[registry['human_validation_set'], 'chunk_id']))
        ]
        n_needed = MIN_POSITIVE - n_positive_in_val
        extra = positive_not_in_val.sample(min(n_needed, len(positive_not_in_val)), random_state=42)
        extra_ids = extra['chunk_id'].tolist()
        registry.loc[registry['chunk_id'].isin(extra_ids), 'human_validation_set'] = True
        registry.loc[registry['chunk_id'].isin(extra_ids), 'oversampled_for'] = flag
        print(f'{flag}: added {len(extra_ids)} oversampled chunks')

print('Final chunk-level validation size:', int(registry['human_validation_set'].sum()))


## Step 4 — Utterance-level subsample

In [ ]:
eligible = registry.loc[registry['human_validation_set']].copy()
target_n = min(50, len(eligible))

utterance_val = (
    eligible.groupby(['conference_id', 'chunk_position'], group_keys=False)
    .apply(lambda g: g.sample(n=max(1, int(round(len(g) * 0.25))), random_state=42))
    .drop_duplicates('chunk_id')
    .head(target_n)
)

registry['utterance_validation_set'] = registry['chunk_id'].isin(utterance_val['chunk_id'])

print('Utterance validation chunks:', int(registry['utterance_validation_set'].sum()))
display(
    registry.loc[registry['utterance_validation_set'], ['chunk_id', 'conference_id', 'chunk_position', 'oversampled_for']]
    .head(20)
)


## Step 5 — Save updated registry

In [ ]:
out_parquet = DATA_DIR / 'chunk_registry_v2.parquet'
out_csv = DATA_DIR / 'chunk_registry_v2.csv'

try:
    registry.to_parquet(out_parquet, index=False)
    print('Saved:', out_parquet)
except Exception as e:
    print(f'Parquet save failed: {type(e).__name__}: {e}')

registry.to_csv(out_csv, index=False)
print('Saved:', out_csv)

display(
    registry[['human_validation_set', 'utterance_validation_set']]
    .sum()
    .rename('n_chunks')
)
